In [1]:
pip install panda

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install bs4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from bs4 import BeautifulSoup
import pandas as pd
import os

d = {'Date': [], 'Headline': [], 'Link': []}
file = "adblnewshtml" + ".html"
# for file in os.listdir("STOCKNEPSE"):
try:
    with open(f"STOCKNEPSE/webScraphtmlfiles/{file}", encoding="utf-8") as f:
        html_doc = f.read()
    
    soup = BeautifulSoup(html_doc, 'html.parser')        
    rows = soup.find('tbody').find_all('tr')
    
    for row in rows:
        cols = row.find_all('td')
        if len(cols) >= 1:
            link_tag = cols[1].find('a')
            d['Date'].append(cols[0].get_text().strip())
            d['Headline'].append(cols[1].get_text().strip())
            d['Link'].append(link_tag['href'].strip())
            # d['Full News'].append(cols[4].get_text().strip())
    
    print(f"{file}: {len(rows)} rows")
    
except Exception as e:
    print(f"{e}")
csv_filename = file.replace('.html', '.csv')
df = pd.DataFrame(data=d)
df.to_csv(f"STOCKNEPSE/NEPSEDATA/{csv_filename}", index=False)

adblnewshtml.html: 496 rows


In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

# Read the CSV file
csv_filename = "adblnewshtml.csv"
df = pd.read_csv(f"STOCKNEPSE/NEPSEDATA/{csv_filename}")

print(f"Total news items: {len(df)}\n")

# Create a new dictionary to store full news content
news_data = {'Date': [], 'Headline': [], 'Link': [], 'Full Content': []}

for index, row in df.iterrows():
    date = row['Date']
    headline = row['Headline']
    link = row['Link']
    
    print(f"\nProcessing {index + 1}/{len(df)}: {headline}")
    
    try:
        # Make request to the link
        response = requests.get(link, timeout=10)
        response.raise_for_status()
        
        # Parse HTML
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Find the newsdetail-content div
        content_div = soup.find('div', id='newsdetail-content')
        
        if content_div:
            # Get all text from the div
            content = content_div.get_text(separator='\n', strip=True)
            print(f"Extracted {len(content)} characters")
        else:
            content = 'Content div not found'
            print("Warning: newsdetail-content div not found")
        
        # Store data
        news_data['Date'].append(date)
        news_data['Headline'].append(headline)
        news_data['Link'].append(link)
        news_data['Full Content'].append(content)
        
    except Exception as e:
        print(f"Error: {e}")
        news_data['Date'].append(date)
        news_data['Headline'].append(headline)
        news_data['Link'].append(link)
        news_data['Full Content'].append(f'Error: {str(e)}')
    
    time.sleep(1)  # Be polite to the server

# Save to new CSV with full content
output_filename = "adblnews_full_content.csv"
df_full = pd.DataFrame(news_data)
df_full.to_csv(f"STOCKNEPSE/NEPSEDATA/{output_filename}", index=False)

print(f"\n{'='*80}")
print(f"Saved full content to: STOCKNEPSE/NEPSEDATA/{output_filename}")
print(f"Total articles processed: {len(df_full)}")

# Display summary
print(f"\nSuccessfully extracted: {len([c for c in news_data['Full Content'] if not c.startswith('Error')])}")
print(f"Errors: {len([c for c in news_data['Full Content'] if c.startswith('Error')])}")

Total news items: 496


Processing 1/496: Bonus Shares of ADBL, ALBSL, SHIVM listed in NEPSE
Extracted 1184 characters

Processing 2/496: Falgun Interest Rate Update: Commercial Banks Revise Fixed Deposit Rates; Majority Keep Rates Unchanged
Extracted 697 characters

Processing 3/496: Commercial Banks Revise Magh Interest Rates; HBL and  MBL Leads with Significant Cuts
Extracted 669 characters

Processing 4/496: Commercial Banks Revise Poush Interest Rates; GBIME, PCBL and RBBL Leads with Significant Cuts
Extracted 688 characters

Processing 5/496: NRB Report Highlights Commercial and Development Banks' Net Profits up to Kartik; GBIME Tops the Charts
Extracted 1555 characters

Processing 6/496: Reminder! Last Trading Day to Secure Dividend of Six Companies
Extracted 4449 characters

Processing 7/496: Agricultural Development Bank Unveils Book Closure Date And Call 19th AGM
Extracted 952 characters

Processing 8/496: ICRA Nepal Revises Ratings: Everest Bank on Withdrawal Notice, Agricul